### Original file [Tutorial: Agentic retrieval using Azure AI Search and Foundry Agent Service](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/agentic-retrieval-pipeline-example/agent-example.ipynb)

# Tutorial: Agentic retrieval using Azure AI Search and Foundry Agent Service

Use this notebook to create an agentic retrieval pipeline built on Azure AI Search and Foundry Agent Service.

In this notebook, you:

1. Create and load an `earth-at-night` search index.

1. Create an `earth-knowledge-source` that targets your index.

1. Create an `earth-knowledge-base` that targets your knowledge source and an LLM for intelligent query planning.

1. Use the knowledge base to fetch, rank, and synthesize relevant information from the index.

1. Create an agent in Foundry Agent Service to determine when queries are needed.

1. Create an MCP tool to orchestrate all requests.

1. Start a chat with the agent.

This notebook is referenced in [Tutorial: Build an end-to-end agentic retrieval solution using Azure AI Search](https://learn.microsoft.com/azure/search/search-agentic-retrieval-how-to-pipeline).

Unlike [Quickstart: Use agentic retrieval in Azure AI Search](https://learn.microsoft.com/azure/search/search-get-started-agentic-retrieval), this quickstart uses Foundry Agent Service to determine whether to retrieve data from the knowledge source and uses an MCP tool for orchestration.

## Prerequisites

+ An Azure AI Search service in any [region that provides agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support).

+ A [Microsoft Foundry project](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/create-projects?view=foundry-classic&tabs=foundry) and resource. When you create a project, the resource is automatically created.

+ A [supported LLM](https://learn.microsoft.com/azure/search/search-agentic-retrieval-how-to-create#supported-models) deployed to your project. This notebook uses `gpt-5-mini`. We recommend a minimum token capacity of 100,000. You can find the LLM's capacity and rate limit in the Foundry portal. If you want vectorization at query time, you should also deploy a text embedding model.

+ [Visual Studio Code](https://code.visualstudio.com/download) with the [Python extension](https://marketplace.visualstudio.com/items?itemName=ms-python.python) and [Jupyter package](https://pypi.org/project/jupyter/).

## Set up connections

Save the `sample.env` file as `.env` and then modify the environment variables to use your Azure endpoints. You need endpoints for:

+ Azure AI Search
+ Azure OpenAI (for the models deployed to your project)
+ Microsoft Foundry project

You also need the resource ID of your project and at least **Azure AI Project Manager** access to the resource to create a connection for authentication. This connection is key based and requires an API key from your search service.

Alternatively, you can omit the key and connect using a managed identity. For this option, you must enable a system-assigned managed identity on your project and assign the project the **Search Index Data Reader** role on your search service.

You can find endpoints for Azure AI Search, Azure OpenAI, and Microsoft Foundry in the [Azure portal](https://portal.azure.com).

## Load connections

We recommend creating a virtual environment to run this sample code. In Visual Studio Code, open the control palette (ctrl-shift-p) to create an environment. This notebook was tested on Python 3.13.7.

After your environment is created, load the environment variables to set up connections and object names.

In [ ]:
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, DefaultAzureCredential, get_bearer_token_provider, InteractiveBrowserCredential

from azure.mgmt.core.tools import parse_resource_id
import os

load_dotenv(override=True) # take environment variables from .env.

project_endpoint = os.getenv("PROJECT_ENDPOINT")
project_resource_id = os.getenv("PROJECT_RESOURCE_ID")
project_connection_name = os.getenv("PROJECT_CONNECTION_NAME", "sharepoint-knowledgeconnection")
agent_model = os.getenv("AGENT_MODEL", "gpt-5-mini")
agent_name = os.getenv("AGENT_NAME", "sharepoint-agent")
endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")

tenant_id = os.getenv("AZURE_TENANT_ID")
credential = AzureCliCredential(tenant_id=tenant_id)
#credential = DefaultAzureCredential()
#credential = InteractiveBrowserCredential(tenant_id=tenant_id)

name_prefix = os.getenv("NAME_PREFIX", "ks-sharepoint-test")
knowledge_source_name = name_prefix
index_name = f"{name_prefix}-index"
base_name = os.getenv("AZURE_SEARCH_AGENT_NAME", "kb-sharepoint-test")

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_gpt_deployment = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT", "gpt-5-mini")
azure_openai_gpt_model = os.getenv("AZURE_OPENAI_GPT_MODEL", "gpt-5-mini")
azure_openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
azure_openai_embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-large")

# Parse the resource ID to extract subscription and other components
parsed_resource_id = parse_resource_id(project_resource_id)
subscription_id = parsed_resource_id['subscription']
resource_group = parsed_resource_id['resource_group']
account_name = parsed_resource_id['name']
project_name = parsed_resource_id['child_name_1']

## Create a search index

This steps create a search index that contains plain text and vector content. You can use an existing index, but it must meet the [criteria for agentic retrieval workloads](https://learn.microsoft.com/azure/search/search-agentic-retrieval-how-to-index). The primary schema requirement is a semantic configuration with a `default_configuration_name`.

We have already created a search index and can skip this section.

## Upload sample documents

This notebook uses data from NASA's Earth at Night e-book. The data is retrieved from the [azure-search-sample-data](https://github.com/Azure-Samples/azure-search-sample-data) repository on GitHub and passed to the search client for indexing.

We have already populated the search index and can skip this section.

## Create a knowledge source

This step creates a knowledge source that targets the index you previously created. In the next step, you create a knowledge base that uses the knowledge source to orchestrate agentic retrieval.


In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, SearchIndexFieldReference,
)
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

ks = SearchIndexKnowledgeSource(
    name=knowledge_source_name,
    description="Knowledge source for SharePoint",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=index_name,
        semantic_configuration_name=f"{name_prefix}-semantic-configuration",
        source_data_fields=[
            SearchIndexFieldReference(name="snippet_parent_id"),
            SearchIndexFieldReference(name="snippet")]
    ),
)

# Use API key if available, otherwise use credential
if search_api_key:
    search_credential = AzureKeyCredential(search_api_key)
else:
    search_credential = credential

index_client = SearchIndexClient(endpoint=endpoint, credential=search_credential)

index_client.create_or_update_knowledge_source(knowledge_source=ks)
print(f"Knowledge source '{ks.name}' created or updated successfully.")

## Create a knowledge base

This step creates a knowledge base, which acts as a wrapper for your knowledge source and LLM deployment.

`EXTRACTIVE_DATA` is the default modality and returns content from your knowledge sources without generative alteration. This is recommended for interaction with Foundry Agent Service

In [ ]:
from azure.search.documents.indexes.models import KnowledgeBase, KnowledgeSourceReference, AzureOpenAIVectorizerParameters, KnowledgeRetrievalOutputMode, KnowledgeRetrievalMediumReasoningEffort, KnowledgeBaseAzureOpenAIModel
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url = azure_openai_endpoint,
    deployment_name = azure_openai_gpt_deployment,
    model_name = azure_openai_gpt_model,
)

kb = KnowledgeBase(
    name=base_name,
    description="Used when searching for automotive knowledge.",
    retrieval_instructions = "You have a helpful car assistant. Look up ks-sharepoint-test to search automotive user manuals.",
    knowledge_sources=[
        KnowledgeSourceReference(name=knowledge_source_name),
    ],
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    models = [KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters = aoai_params)],
    retrieval_reasoning_effort=KnowledgeRetrievalMediumReasoningEffort()
)

# Use API key if available, otherwise use credential
if search_api_key:
    search_credential = AzureKeyCredential(search_api_key)
else:
    search_credential = credential

index_client = SearchIndexClient(endpoint=endpoint, credential=search_credential)

index_client.create_or_update_knowledge_base(knowledge_base=kb)
print(f"Knowledge base '{kb.name}' created or updated successfully")

mcp_endpoint = f"{endpoint}/knowledgebases/{base_name}/mcp?api-version=2025-11-01-Preview"

<h3> Test Kowkledge kb with Index Source</h3>

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest, RemoteSharePointKnowledgeSourceParams, SearchIndexKnowledgeSourceParams, WebKnowledgeSourceParams

kb_client = KnowledgeBaseRetrievalClient(endpoint =endpoint, knowledge_base_name = base_name, credential = search_credential)

request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role = "assistant",
            content = [KnowledgeBaseMessageTextContent(text = "You are the car manual assistant.")]
        ),
        KnowledgeBaseMessage(
            role = "user",
            content = [KnowledgeBaseMessageTextContent(text = "What is the cruise control in Prius?")]
        ),
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name = knowledge_source_name,
            include_references = True,
            include_reference_source_data = True,
            always_query_source = True,
        )
    ],
    include_activity = True,
)

# Permission-filtered search requires the caller's Entra ID token.
# get_bearer_token_provider uses DefaultAzureCredential (logged-in user via Azure CLI / VS Code).
user_token = get_bearer_token_provider(credential, "https://search.azure.com/.default")()

result = kb_client.retrieve(request, x_ms_query_source_authorization=user_token)
print(result.response[0].content[0].text)

In [ ]:
import json

# Print the activity details in readable format
print("=== Activity Details ===\n")
if hasattr(result, 'activity') and result.activity:
    # Convert to dict if it has as_dict method, otherwise use vars
    if hasattr(result.activity, 'as_dict'):
        activity_dict = result.activity.as_dict()
    elif hasattr(result.activity, '__dict__'):
        activity_dict = vars(result.activity)
    else:
        activity_dict = result.activity
    
    print(json.dumps(activity_dict, indent=2, ensure_ascii=False, default=str))
else:
    print("No activity found")

# Print references in readable format
print("\n=== References ===\n")
if hasattr(result, 'references') and result.references:
    for i, ref in enumerate(result.references):
        print(f"Reference {i+1}:")
        print("-" * 40)
        if hasattr(ref, 'as_dict'):
            ref_dict = ref.as_dict()
        elif hasattr(ref, '__dict__'):
            ref_dict = vars(ref)
        else:
            ref_dict = ref
        print(json.dumps(ref_dict, indent=2, ensure_ascii=False, default=str))
        print()
else:
    print("No references found")

## List existing agents

In Foundry Agent Service, an agent is a smart micro-service that can do RAG. The purpose of this agent is to decide when to send a query to the agentic retrieval pipeline.

In [ ]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

list(project_client.agents.list())

## Create an MCP tool connection to AI Search MCP Server

In Microsoft Foundry, you must create a connection to authenticate to your MCP tool.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import ConnectionPropertiesV2BasicResource, CustomKeysConnectionProperties, CustomKeys
import requests
from azure.identity import get_bearer_token_provider

if search_api_key:
    connection = ConnectionPropertiesV2BasicResource(
        properties=CustomKeysConnectionProperties(
            category="RemoteTool",
            target=mcp_endpoint,
            is_shared_to_all=True,
            metadata={ "ApiType": "Azure" },
            credentials=CustomKeys(
                keys={ "api-key": search_api_key }
            )
        )
    )

    mgmt_client = CognitiveServicesManagementClient(credential, subscription_id)
    resource = mgmt_client.project_connections.create(
        resource_group_name=resource_group,
        account_name=account_name,
        project_name=project_name,
        connection_name=project_connection_name,
        connection=connection
    )

    print(f"Connection '{resource.name}' created or updated successfully.")
else:
    # Requires the Foundry project to have Search Index Data Reader role on the search service
    bearer_token_provider = get_bearer_token_provider(credential, "https://management.azure.com/.default")
    headers = {
        "Authorization": f"Bearer {bearer_token_provider()}",
    }
    response = requests.put(
        f"https://management.azure.com{project_resource_id}/connections/{project_connection_name}?api-version=2025-10-01-preview",
        headers=headers,
        json={
            "name": project_connection_name,
            "type": "Microsoft.MachineLearningServices/workspaces/connections",
            "properties": {
                "authType": "ProjectManagedIdentity",
                "category": "RemoteTool",
                "target": mcp_endpoint,
                "isSharedToAll": True,
                "audience": "https://search.azure.com/",
                "metadata": { "ApiType": "Azure" }
            }
        }
    )
    response.raise_for_status()
    print(f"Connection '{project_connection_name}' created or updated successfully.")

## Create an agent with a knowledge base

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

instructions = """
You are a car assistant that provides information about user's car. Please utilize the knowledge base that contains car information.
Answer in the same language as the user's query.
"""
mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=project_connection_name
)
agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=instructions,
        tools=[mcp_kb_tool]
    )
)


print(f"AI agent '{agent_name}' created or updated successfully")

## Start a chat with the agent

In [ ]:
# Get the OpenAI client for responses and conversations
openai_client = project_client.get_openai_client()

conversation = openai_client.conversations.create()

user_token = get_bearer_token_provider(credential, "https://search.azure.com/.default")()

# Send initial request that will trigger the MCP tool.
# x-ms-query-source-authorization is propagated to the knowledge base MCP tool call
# so that permission-filtered search works with the caller's Entra ID token.
response = openai_client.responses.create(
    conversation=conversation.id,
    input="""
        What is the cruise control in Prius?
    """,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    extra_headers={"x-ms-query-source-authorization": user_token},
)

print(f"Response: {response.output_text}")

## Clean up objects and resources

If you no longer need Azure AI Search or Microsoft Foundry, delete the resources from your Azure subscription. You can also start over by deleting individual objects.

### Delete the agent

In [ ]:
project_client.agents.delete_version(agent.name, agent.version)
print(f"AI agent '{agent.name}' version '{agent.version}' deleted successfully")

### Delete the knowledge base

In [ ]:
index_client.delete_knowledge_base(base_name)
print(f"Knowledge base '{base_name}' deleted successfully")

### Delete the knowledge source

In [ ]:
index_client.delete_knowledge_source(knowledge_source=knowledge_source_name) # This is new feature in 2025-08-01-Preview api version
print(f"Knowledge source '{knowledge_source_name}' deleted successfully.")


### Delete the search index

In [ ]:
index_client.delete_index(index)
print(f"Index '{index_name}' deleted successfully")